# SchoolBridge — LayoutXLM Train/Test Split 검증 (A안: 6장)

**목적**: 1장 overfit 한계 극복. 학습 안 본 통신문에 일반화 가능한지 검증.

**Setup**:
- 학습 4장 + 테스트 2장 (총 6장)
- 학습: 구강검진 · 서귀포 · 도서관 · 어린이날
- 테스트: 인플루엔자 · 000002331245

**Colab 세팅**: 런타임 → T4 GPU

## 1. 환경 + 설치

In [ ]:
!pip install -q transformers sentencepiece pdfplumber pymupdf pillow
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'
print("설치 완료")

In [ ]:
import torch
import detectron2
from transformers import LayoutXLMProcessor, LayoutLMv2ForTokenClassification
print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
print("detectron2:", detectron2.__version__)

## 2. PDF 6장 업로드

`backend/data/` 에서 6장 (업로드 순서가 학습/테스트 순서):

**학습 (4장)**:
1. `2026 2,3,5,6학년 구강검진 실시안내.pdf`
2. `2026년+5월+서귀포외국문화학습관+토요프로그램+추가+모집+안내.pdf`
3. `2025. 겨울방학 도서관 이용 및 독서캠프 신청 안내.pdf`
4. `2026 북부과학교육관 어린이날 행사 안내 가정통신문.pdf`

**테스트 (2장)**:
5. `인플루엔자예방접종접종안내(2019).pdf`
6. `000002331245_20260510220822.pdf`

In [ ]:
from google.colab import files
from pathlib import Path
uploaded = files.upload()
pdf_paths = [Path(name) for name in uploaded.keys()
             if Path(name).suffix.lower() == ".pdf"]
print(f"업로드 PDF {len(pdf_paths)}장:")
for i, p in enumerate(pdf_paths):
    role = "학습" if i < 4 else "테스트"
    print(f"  [{i}] ({role}) {p.name}: {p.stat().st_size // 1024} KB")

## 3. 추출 + 페이지 이미지 (6장 모두)

In [ ]:
from dataclasses import dataclass
from typing import List, Tuple
import pdfplumber
import fitz
from PIL import Image

@dataclass
class TextSpan:
    text: str
    bbox: Tuple[float, float, float, float]
    page: int = 0

def extract_pdf(path: Path) -> List[TextSpan]:
    spans = []
    with pdfplumber.open(path) as pdf:
        for page_idx, page in enumerate(pdf.pages):
            W, H = page.width, page.height
            words = page.extract_words(
                use_text_flow=True, keep_blank_chars=False,
                x_tolerance=3, y_tolerance=3,
            )
            for w in words:
                bbox = (w["x0"]/W, w["top"]/H, w["x1"]/W, w["bottom"]/H)
                spans.append(TextSpan(text=w["text"], bbox=bbox, page=page_idx))
    return spans

def render_pdf_page(path: Path, page_idx: int = 0, dpi: int = 150) -> Image.Image:
    doc = fitz.open(path)
    page = doc[page_idx]
    pix = page.get_pixmap(dpi=dpi)
    img = Image.frombytes("RGB", (pix.width, pix.height), pix.samples)
    doc.close()
    return img

all_docs = []
for p in pdf_paths:
    spans = [s for s in extract_pdf(p) if s.page == 0]
    img = render_pdf_page(p, page_idx=0, dpi=150)
    all_docs.append({"name": p.name, "spans": spans, "image": img})
    print(f"📄 {p.name}: {len(spans)} tokens, image {img.size}")

## 4. 각 PDF의 spans 출력 — 라벨링용 (6장 모두)

셀 출력 보면서 셀 11에서 각 PDF의 sentence_ranges 작성.

In [ ]:
for doc_idx, doc in enumerate(all_docs):
    role = "학습" if doc_idx < 4 else "테스트"
    print(f"\n{'='*70}")
    print(f"📄 [{doc_idx}] ({role}) {doc['name']} — {len(doc['spans'])} tokens")
    print(f"{'='*70}")
    for i, s in enumerate(doc['spans']):
        print(f"  [{i:3d}] (y={s.bbox[1]:.3f}) {s.text!r}")

## 5. 라벨링 — 각 PDF별 sentence_ranges (6개 모두)

**팁**: 새 통신문은 의미 묶음 5~10개 정도로 빨리 라벨. 정밀할 필요 X — 큰 흐름만.

- id=0: 발송 정보
- id=1: 제목 + 인사
- id=2: 본문
- id=3, 4, 5...: 1번 항목, 2번 항목, ...
- id=10, 11: 표 영역 (분리되는 경우)

**페이지마다 id 독립** (id=0이 PDF마다 다른 의미여도 OK).

In [ ]:
all_labels = [None] * 6

# [0] 학습 — 구강검진 (이미 작성됨)
all_labels[0] = [
    (0,   9,   0, "발송 정보"),
    (10,  15,  1, "제목 + 인사"),
    (16,  60,  2, "본문 안내문"),
    (61,  75,  3, "1. 검진 대상"),
    (76,  95,  4, "2. 검진 기간"),
    (96,  110, 5, "3. 검진 항목"),
    (111, 125, 6, "4. 검사 비용"),
    (126, 145, 7, "5. 검진 기관"),
    (146, 168, 10, "진심담은치과 묶음"),
    (169, 186, 11, "청담i치과 묶음"),
    (187, 196, 12, "발급일자 + 학교장"),
    (197, 210, 13, "절취선 + 확인서"),
    (211, 230, 14, "확인서 양식"),
]

# [1] 학습 — 서귀포 토요프로그램 (셀 4 출력 보고 작성)
all_labels[1] = [
    # (start, end, sid, "주석") — 채우기
]

# [2] 학습 — 도서관 이용 (셀 4 출력 보고 작성)
all_labels[2] = [
    # (start, end, sid, "주석") — 채우기
]

# [3] 학습 — 어린이날 행사 (셀 4 출력 보고 작성)
all_labels[3] = [
    # (start, end, sid, "주석") — 채우기
]

# [4] 테스트 — 인플루엔자 예방접종 (셀 4 출력 보고 작성)
all_labels[4] = [
    # (start, end, sid, "주석") — 채우기
]

# [5] 테스트 — 000002331245 (셀 4 출력 보고 작성)
all_labels[5] = [
    # (start, end, sid, "주석") — 채우기
]

# word_labels 생성
from collections import Counter

MAX_SENT_ID = 20

for doc_idx, (doc, ranges) in enumerate(zip(all_docs, all_labels)):
    if not ranges:
        print(f"⚠️ [{doc_idx}] {doc['name']}: 라벨링 비어있음 — 채워야 함")
        doc['word_labels'] = None
        continue
    
    spans = doc['spans']
    word_labels = [-1] * len(spans)
    for start, end, sid, _ in ranges:
        for i in range(start, min(end + 1, len(word_labels))):
            word_labels[i] = sid
    max_sid = max(sid for _, _, sid, _ in ranges)
    word_labels = [w if w >= 0 else (max_sid + 1) for w in word_labels]
    doc['word_labels'] = word_labels
    doc['ranges'] = ranges
    role = "학습" if doc_idx < 4 else "테스트"
    print(f"[{doc_idx}] ({role}) {doc['name']}: {len(set(word_labels))}개 묶음, {len(word_labels)} tokens")

## 6. LayoutXLM 모델 로드

In [ ]:
MODEL_ID = "microsoft/layoutxlm-base"

processor = LayoutXLMProcessor.from_pretrained(MODEL_ID, apply_ocr=False)
model = LayoutLMv2ForTokenClassification.from_pretrained(
    MODEL_ID,
    num_labels=MAX_SENT_ID,
    id2label={i: f"SENT_{i}" for i in range(MAX_SENT_ID)},
    label2id={f"SENT_{i}": i for i in range(MAX_SENT_ID)},
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"Model: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M params | Device: {device}")

def to_layoutxlm_inputs(image, spans, word_labels=None):
    words = [s.text for s in spans]
    boxes = []
    for s in spans:
        x0, y0, x1, y1 = s.bbox
        boxes.append([
            max(0, min(1000, int(x0 * 1000))),
            max(0, min(1000, int(y0 * 1000))),
            max(0, min(1000, int(x1 * 1000))),
            max(0, min(1000, int(y1 * 1000))),
        ])
    encoded = processor(
        image, words, boxes=boxes,
        return_tensors="pt", truncation=True,
        padding="max_length", max_length=512,
    )
    if word_labels is not None:
        word_ids = encoded.word_ids()
        token_labels = []
        for wid in word_ids:
            token_labels.append(-100 if wid is None else word_labels[wid])
        encoded["labels"] = torch.tensor([token_labels])
    return encoded

## 7. Train/Test Split — 4장 학습 / 2장 테스트

In [ ]:
TRAIN_IDX = [0, 1, 2, 3]
TEST_IDX = [4, 5]

train_docs = [all_docs[i] for i in TRAIN_IDX if all_docs[i].get('word_labels')]
test_docs = [all_docs[i] for i in TEST_IDX if all_docs[i].get('word_labels')]

print(f"학습셋 {len(train_docs)}장:")
for d in train_docs:
    print(f"  - {d['name']}")
print(f"\n테스트셋 {len(test_docs)}장:")
for d in test_docs:
    print(f"  - {d['name']}")

## 8. Fine-tune (학습셋 4장)

In [ ]:
model.train()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5, weight_decay=0.01)

train_inputs = []
for d in train_docs:
    enc = to_layoutxlm_inputs(d['image'], d['spans'], word_labels=d['word_labels'])
    train_inputs.append({k: v.to(device) for k, v in enc.items()})

print(f"=== Fine-tune ({len(train_docs)}장, 50 epoch) ===")
for epoch in range(50):
    total_loss = 0.0
    for inp in train_inputs:
        optimizer.zero_grad()
        outputs = model(**inp)
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(train_inputs)
    if epoch % 5 == 0 or epoch == 49:
        print(f"  Epoch {epoch+1:2d}: avg loss = {avg_loss:.4f}")

## 9. 테스트셋 평가 (2장)

In [ ]:
from collections import defaultdict

model.eval()
total_correct = 0
total_tokens = 0

for doc in test_docs:
    encoded = to_layoutxlm_inputs(doc['image'], doc['spans'])
    inputs = {k: v.to(device) for k, v in encoded.items()}
    with torch.no_grad():
        outputs = model(**inputs)
    predictions = outputs.logits.argmax(-1)[0].tolist()
    
    word_ids = encoded.word_ids()
    word_preds = []
    for w_idx in range(len(doc['spans'])):
        tok_preds = [predictions[t] for t, w in enumerate(word_ids) if w == w_idx]
        if tok_preds:
            word_preds.append(Counter(tok_preds).most_common(1)[0][0])
        else:
            word_preds.append(-1)
    
    correct = sum(1 for p, g in zip(word_preds, doc['word_labels']) if p == g)
    acc = correct / len(word_preds) * 100
    total_correct += correct
    total_tokens += len(word_preds)
    
    print(f"\n{'='*70}")
    print(f"📄 테스트: {doc['name']}")
    print(f"   정확도: {correct} / {len(word_preds)} = {acc:.1f}%")
    print(f"{'='*70}")
    
    # sentence_list 생성
    groups = defaultdict(list)
    first_app = {}
    for idx, (span, sid) in enumerate(zip(doc['spans'], word_preds)):
        groups[sid].append(span)
        first_app.setdefault(sid, idx)
    sorted_sids = sorted(groups.keys(), key=lambda s: first_app[s])
    
    print(f"\n예측 sentence_list ({len(sorted_sids)}개 묶음):")
    for sid in sorted_sids:
        text = " ".join(s.text for s in groups[sid])
        print(f"\n[묶음 id={sid}]")
        print(f"  {text[:200]}")

print(f"\n\n{'='*70}")
print(f"🎯 전체 테스트셋 정확도: {total_correct} / {total_tokens} = {total_correct/total_tokens*100:.1f}%")
print(f"{'='*70}")

## 10. 결과 해석

| 테스트 정확도 | 의미 |
|---|---|
| **70%+** | 4장 학습으로도 일반화 → 본격 라벨링 (100~300장) 가치 큼 |
| 50~70% | 가능성 있음, 학습 데이터 확대 필수 |
| 50% 이하 | 1장 overfit 한계 그대로 — 본격 라벨링 전 다른 검증 필요 |

**1장 overfit 96% 와 비교**: 떨어진 격차가 진짜 일반화 격차. 30%+ 떨어지면 학습 데이터 많이 필요.